In [2]:
import requests
import spacy

API_KEY  = "27d4364fba3f7caee94dcd212a4b7199"
BASE_URL = "https://api.themoviedb.org/3"

## NER Example

In [4]:
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Fetch movie overview from TMDB
def fetch_movie_overview(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {"api_key": API_KEY, "language": "en-US"}
    r = requests.get(url, params=params, timeout=10)
    data = r.json()
    return data.get("title", ""), data.get("overview", "")

# Test NER on a few movies
test_movies = [27205, 157336, 603, 550, 238]  # Inception, Interstellar, Matrix, Fight Club, Godfather

print(" NER Results\n")
for movie_id in test_movies:
    title, overview = fetch_movie_overview(movie_id)
    if not overview:
        continue
    
    doc = nlp(overview)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    
    print(f"Movie: {title}")
    print(f"Overview: {overview[:150]}...")
    print(f"Entities: {entities}")
    print()

 NER Results

Movie: Inception
Overview: Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as pa...
Entities: []

Movie: Interstellar
Overview: The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the va...
Entities: []

Movie: The Matrix
Overview: Set in the 22nd century, The Matrix tells the story of a computer hacker who joins a group of underground insurgents fighting the vast and powerful co...
Entities: [('the 22nd century', 'DATE')]

Movie: Fight Club
Overview: A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches o...
Entities: []

Movie: The Godfather
Overview: Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, V

## Analysis of Ambiguous Cases

In [5]:
print("=== 3 Ambiguity Cases ===\n")

# Case 1: "The Matrix" - Is "Matrix" a movie title or a mathematical matrix?
title, overview = fetch_movie_overview(603)
doc = nlp(overview)
print("Case 1: 'The Matrix'")
print(f"Text: {overview[:200]}")
print("Ambiguity: 'The Matrix' is not recognized as a film title by NER,")
print("because 'matrix' is also a common noun in mathematics/computing.")
print(f"NER result: {[(ent.text, ent.label_) for ent in doc.ents]}\n")

# Case 2: "Vito Corleone" misclassified as ORG instead of PERSON
title, overview = fetch_movie_overview(238)
doc = nlp(overview)
print("Case 2: 'Vito Corleone' misclassified")
print(f"Text: {overview[:200]}")
print("Ambiguity: 'Vito Corleone' should be PERSON but was tagged as ORG,")
print("because 'Corleone' also refers to a place/family organization.")
print(f"NER result: {[(ent.text, ent.label_) for ent in doc.ents]}\n")

# Case 3: "Michael" - Is it a person's name or a character name?
print("Case 3: 'Michael' - incomplete entity recognition")
print("Ambiguity: 'Michael' is recognized as PERSON but without surname,")
print("making identity resolution impossible without context.")
print("This shows the limitation of NER without coreference resolution.\n")

=== 3 Ambiguity Cases ===

Case 1: 'The Matrix'
Text: Set in the 22nd century, The Matrix tells the story of a computer hacker who joins a group of underground insurgents fighting the vast and powerful computers who now rule the earth.
Ambiguity: 'The Matrix' is not recognized as a film title by NER,
because 'matrix' is also a common noun in mathematics/computing.
NER result: [('the 22nd century', 'DATE')]

Case 2: 'Vito Corleone' misclassified
Text: Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his y
Ambiguity: 'Vito Corleone' should be PERSON but was tagged as ORG,
because 'Corleone' also refers to a place/family organization.
NER result: [('the years 1945 to 1955', 'DATE'), ('Italian', 'NORP'), ('Vito Corleone', 'ORG'), ('Michael', 'PERSON')]

Case 3: 'Michael' - incomplete entity recognition
Ambiguity: 'Michael' is recognized as PERSON